In [1]:
import os
from typing import Annotated, TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

load_dotenv()

True

In [ ]:
# 1. Định nghĩa State
class TuVanState(TypedDict):
    messages: Annotated[list, add_messages]
    ten_khach: str
    san_pham_quan_tam: str

In [ ]:
# 2. Khởi tạo Graph Builder
graph_builder = StateGraph(TuVanState)

In [ ]:
# 3. Tao Node
# Node 1 - Trích xuất thông tin
def trich_xuat_thong_tin(state: TuVanState) -> dict:
    # Lấy tin nhắn mới nhất của khách
    tin_nhan_moi = state["messages"][-1].content
    ten_hien_tai = state.get("ten_khach", "")
    
    # Tìm tên khách nếu khách tự giới thiệu
    ten_moi = ten_hien_tai
    cac_ten_pho_bien = ["mình là", "tôi là", "em là", "anh là", "chị là", "tên mình là", "tên em là"]
    for mau in cac_ten_pho_bien:
        if mau in tin_nhan_moi.lower():
            # Lấy từ ngay sau cụm "tên gợi ý"
            vi_tri = tin_nhan_moi.lower().find(mau) + len(mau)
            phan_con_lai = tin_nhan_moi[vi_tri:].strip().split()
            if phan_con_lai:
                ten_moi = phan_con_lai[0].capitalize()
            break
    
    # Chỉ trả về field thay đổi
    return {"ten_khach": ten_moi}

In [5]:
# Node 2 - Gọi LLM để tạo câu trả lời
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Node 2: Tư vấn viên AI
def tu_van_vien(state: TuVanState) -> dict:
    ten_khach = state.get("ten_khach", "")
    
    # Xây system prompt cá nhân hóa theo những gì biết về khách
    loi_chao = f"Chào {ten_khach}! " if ten_khach else ""
    
    system_prompt = f"""Bạn là tư vấn viên của shop thời trang LUNA - chuyên thời trang nữ cao cấp tại TP.HCM.
    
Phong cách giao tiếp: thân thiện, chuyên nghiệp, dùng ngôn ngữ tự nhiên của người Việt.
Luôn xưng hô phù hợp (anh/chị/em) tùy theo context.
Trả lời ngắn gọn, đúng trọng tâm — không quá 3-4 câu mỗi lần.
Nếu khách hỏi về sản phẩm cụ thể mà bạn không chắc, hãy hỏi thêm để tư vấn chính xác hơn.
Nếu đây là lần đầu gặp khách: {loi_chao if loi_chao else "hãy chào hỏi thân thiện."}"""
    
    # Tạo danh sách tin nhắn để gửi cho LLM
    tin_nhan_cho_llm = [SystemMessage(content=system_prompt)] + state["messages"]
    
    # Gọi LLM
    phan_hoi = llm.invoke(tin_nhan_cho_llm)
    
    # Trả về — LangGraph sẽ tự add_messages để nối vào list
    return {"messages": [phan_hoi]}

In [ ]:
# 4. Tạo Edges
graph_builder.add_node("trich_xuat", trich_xuat_thong_tin)
graph_builder.add_node("tu_van", tu_van_vien)

graph_builder.add_edge(START, "trich_xuat")
graph_builder.add_edge("trich_xuat", "tu_van")
graph_builder.add_edge("tu_van", END)

In [ ]:
# 5. Compile Graph
graph = graph_builder.compile()

In [ ]:
# Chay graph
def chat(tin_nhan_nguoi_dung: str, lich_su: list) -> tuple[str, list]:
    messages = []
    for role, content in lich_su:
        if role == "user":
            messages.append(HumanMessage(content=content))
        else:
            messages.append(AIMessage(content=content))
    
    messages.append(HumanMessage(content=tin_nhan_nguoi_dung))

    state_dau_vao = {
        "messages": messages,
        "ten_khach": "",
        "san_pham_quan_tam": ""
    }

    ket_qua = graph.invoke(state_dau_vao)

    cau_tra_loi = ket_qua["messages"][-1].content

    lich_su_moi = lich_su + [
        ("user", tin_nhan_nguoi_dung),
        ("assistant", cau_tra_loi)
    ]

    return cau_tra_loi, lich_su_moi

In [13]:
import gradio as gr

def gradio_chat(user_message, history):

    # Convert sang format backend của bạn
    lich_su = []
    for msg in history:
        if msg["role"] == "user":
            lich_su.append(("user", msg["content"]))
        else:
            lich_su.append(("assistant", msg["content"]))

    # Gọi backend
    response, new_history = chat(user_message, lich_su)
    
    gradio_history = []
    for role, content in new_history:
        gradio_history.append({
            "role": role,
            "content": content
        })

    return "", gradio_history


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 💬 LUNA - Tư vấn thời trang")

    chatbot = gr.Chatbot()

    msg = gr.Textbox(
        placeholder="Nhập tin nhắn của bạn...",
        show_label=False
    )

    clear = gr.Button("🗑️ Xóa chat")

    msg.submit(
        gradio_chat,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    clear.click(lambda: [], None, chatbot)


demo.launch()

/var/folders/vm/lyy3xj0j4xdcmhhqjpz379n40000gn/T/ipykernel_62605/3086614154.py:26: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
